# Gemini File Search — Managed RAG: PyTM Threat Model

**Notebook:** `notebooks/gemini_filesearch_pytm_threat_model.ipynb`
**Project:** [Shift-Left Threat Modelling](https://github.com/carlchinx/shift-left-threat-modelling)

This notebook builds a PyTM threat model for a Gemini File Search (managed RAG) API,
renders a Data Flow Diagram (DFD), generates a sequence diagram (PlantUML source),
and emits a Markdown threat report into `out/`.

It is **cross-platform** (Windows / macOS / Linux / Colab). The Linux-only
`apt-get` / `wget` cells from the original Colab version have been replaced with
pure-Python prerequisite checks. Python interpreter is **not downgraded** — the
notebook runs on Python 3.11+.

---

**Author:** Dr. Charles C. Phiri, CITP, Senior IEEE Member, Fellow (ICTAM)
**Affiliation:** Independent Researcher / ICTAM Fellow
**License:** MIT — © 2026 Charles C. Phiri (see [`LICENSE`](../LICENSE))

## License (MIT)

```text
MIT License

Copyright (c) 2026 Charles C. Phiri

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND...
```

Full text: [`LICENSE`](../LICENSE).

## 1. Prerequisites & setup

Installs the Python packages used by this notebook. Native binaries
(`graphviz` / Java for PlantUML) are **detected**, not installed — install them
via your OS package manager if missing:

| OS | Graphviz | Java |
|---|---|---|
| Windows (winget) | `winget install Graphviz.Graphviz` | `winget install EclipseAdoptium.Temurin.21.JDK` |
| macOS (brew) | `brew install graphviz` | `brew install temurin` |
| Debian/Ubuntu | `sudo apt-get install graphviz default-jre` | (same) |
| Colab | `!apt-get install -y graphviz openjdk-11-jre-headless` | (same) |

In [ ]:
import sys, subprocess

# Install Python deps (idempotent, quiet).
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "pytm>=1.3.1", "graphviz>=0.20", "jinja2>=3.1",
])
print(f"Python: {sys.version.split()[0]}")

In [ ]:
import os, shutil, urllib.request
from pathlib import Path

# Output directory (relative to the repo root when run from notebooks/).
OUT_DIR = Path(__file__).resolve().parent.parent / "out" if "__file__" in globals() else Path.cwd().parent / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUT_DIR}")

# Detect graphviz `dot` binary (required for DFD rendering).
HAS_DOT = shutil.which("dot") is not None
print(f"graphviz `dot` binary: {'FOUND' if HAS_DOT else 'MISSING (DFD render will be skipped)'}")

# Detect Java (required only for PlantUML sequence diagram render).
HAS_JAVA = shutil.which("java") is not None
print(f"`java` binary:         {'FOUND' if HAS_JAVA else 'MISSING (sequence PNG will be skipped, .puml still saved)'}")

# Optional: download PlantUML jar locally if Java is available and jar absent.
PLANTUML_JAR = OUT_DIR / "plantuml.jar"
if HAS_JAVA and not PLANTUML_JAR.exists():
    try:
        url = "https://github.com/plantuml/plantuml/releases/download/v1.2024.7/plantuml-1.2024.7.jar"
        print(f"Downloading PlantUML jar -> {PLANTUML_JAR} ...")
        urllib.request.urlretrieve(url, PLANTUML_JAR)
        print("Downloaded.")
    except Exception as e:
        print(f"PlantUML download skipped: {e}")

## 2. Build the PyTM threat model

Three trust boundaries:

* **Developer Environment** — caller / SDK
* **Google Gemini API** — public API gateway and auth
* **Google Internal Systems** — file store, embedder, vector store, retriever, LLM

Each component declares CIA properties and each dataflow is encrypted (HTTPS
where applicable) so PyTM only flags genuinely missing controls.

In [ ]:
import sys
sys.argv = [""]  # PyTM expects argparse to succeed in non-CLI environments.

from pytm import TM, Boundary, Actor, Server, Lambda, Datastore, Dataflow

tm = TM("Gemini File Search - Managed RAG API")
tm.description = "Threat model for Gemini File Search RAG API"

# Boundaries
user_boundary = Boundary("Developer Environment")
google_api_boundary = Boundary("Google Gemini API")
internal_managed_boundary = Boundary("Google Internal Systems")

# Actors / components
developer = Actor("Developer / API Client"); developer.inBoundary = user_boundary

client_app = Server("Gemini SDK / REST Client"); client_app.inBoundary = user_boundary
client_app.providesConfidentiality = True; client_app.providesIntegrity = True

gemini_api = Server("Gemini API Gateway"); gemini_api.inBoundary = google_api_boundary
gemini_api.providesAuthentication = True
gemini_api.providesConfidentiality = True
gemini_api.providesIntegrity = True

auth_service = Server("Auth & Billing Service"); auth_service.inBoundary = google_api_boundary
auth_service.providesAuthentication = True

file_store = Datastore("User File Store"); file_store.inBoundary = internal_managed_boundary
file_store.isEncrypted = True
file_store.providesConfidentiality = True
file_store.providesIntegrity = True

embedder = Lambda("Embedding Service"); embedder.inBoundary = internal_managed_boundary
embedder.providesConfidentiality = True

vector_store = Datastore("Vector Store"); vector_store.inBoundary = internal_managed_boundary
vector_store.isEncrypted = True
vector_store.providesConfidentiality = True

retriever = Lambda("Context Injection Engine"); retriever.inBoundary = internal_managed_boundary
retriever.providesIntegrity = True

llm = Server("Gemini LLM"); llm.inBoundary = internal_managed_boundary
llm.providesConfidentiality = True

# Dataflows (all encrypted; protocol set on public hops)
def flow(src, dst, label, *, https=False, auth_src=False):
    df = Dataflow(src, dst, label)
    df.isEncrypted = True
    if https:
        df.protocol = "HTTPS"
    if auth_src:
        df.authenticatesSource = True
    return df

flow(developer, client_app, "API key/token", https=True)
flow(client_app, auth_service, "Token validation", auth_src=True)
flow(developer, client_app, "File upload")
flow(client_app, gemini_api, "File to API", https=True)
flow(gemini_api, file_store, "Store file")
flow(file_store, embedder, "Trigger embedding")
flow(embedder, vector_store, "Store vectors")
flow(developer, client_app, "User prompt")
flow(client_app, gemini_api, "Prompt to API", https=True)
flow(gemini_api, retriever, "Query vectors")
flow(vector_store, retriever, "Relevant chunks")
flow(retriever, llm, "Context + prompt")
flow(llm, gemini_api, "LLM response")
flow(gemini_api, client_app, "API response", https=True)
flow(client_app, developer, "Display response")

tm.process()
print(f"Model built. Threats identified: {len(tm._threats)}")

## 3. Render the Data Flow Diagram (DFD)

Writes `out/gemini_dfd.png` if the `dot` binary is on PATH; otherwise saves
the Graphviz source to `out/gemini_dfd.dot` for manual rendering.

In [ ]:
from IPython.display import Image, display
from graphviz import Source

dfd_dot = tm.dfd()
(OUT_DIR / "gemini_dfd.dot").write_text(dfd_dot, encoding="utf-8")

if HAS_DOT:
    src = Source(dfd_dot, directory=str(OUT_DIR), filename="gemini_dfd", format="png")
    png_path = src.render(cleanup=True)
    print(f"Wrote {png_path}")
    display(Image(png_path))
else:
    print(f"dot binary not found; saved DOT source only -> {OUT_DIR / 'gemini_dfd.dot'}")

## 4. Sequence diagram (PlantUML)

Saves `out/gemini_sequence.puml`. If Java + the PlantUML jar are available,
also renders `out/gemini_sequence.png`.

In [ ]:
puml_path = OUT_DIR / "gemini_sequence.puml"
puml_path.write_text(tm.seq(), encoding="utf-8")
print(f"Wrote {puml_path}")

if HAS_JAVA and PLANTUML_JAR.exists():
    try:
        subprocess.run(
            ["java", "-jar", str(PLANTUML_JAR), "-tpng", str(puml_path), "-o", str(OUT_DIR)],
            check=True, capture_output=True,
        )
        seq_png = OUT_DIR / "gemini_sequence.png"
        if seq_png.exists():
            print(f"Wrote {seq_png}")
            display(Image(str(seq_png)))
    except subprocess.CalledProcessError as e:
        print(f"PlantUML render failed: {e.stderr.decode(errors='ignore')[:200]}")
else:
    print("Skipping PNG render (need Java + plantuml.jar). PUML source saved.")

## 5. Generate the Markdown threat report

Renders `out/gemini_threat_report.md` from a Jinja2 template based on the
PyTM-evaluated threats.

In [ ]:
from jinja2 import Template
from IPython.display import Markdown

template = Template("""# Threat Model Report: {{ tm.name }}

{{ tm.description }}

## Findings ({{ threats|length }})

{% for f in threats %}### {{ f.id }} — {{ f.description }}

- **Severity:** {{ f.severity }}
- **Target:** {{ f.target }}
- **Details:** {{ f.details }}
- **Mitigations:** {{ f.mitigations }}

{% endfor %}
""")

report_md = template.render(tm=tm, threats=tm._threats)
report_path = OUT_DIR / "gemini_threat_report.md"
report_path.write_text(report_md, encoding="utf-8")
print(f"Wrote {report_path}")

high   = sum(1 for t in tm._threats if str(getattr(t, "severity", "")).lower() == "high")
medium = sum(1 for t in tm._threats if str(getattr(t, "severity", "")).lower() == "medium")
low    = sum(1 for t in tm._threats if str(getattr(t, "severity", "")).lower() == "low")
print(f"Severity breakdown -- high: {high}, medium: {medium}, low: {low}")

display(Markdown(report_md))

---

© 2026 Dr. Charles C. Phiri — released under the [MIT License](../LICENSE).